# 🧱 Arrays — the second dimension

**Time:** ~35 minutes &nbsp;•&nbsp; **Goal:** stack 254 single trials into one **table** of numbers, and average down it to reveal a signal no single trial shows.

You finished notebook 2 holding a *list* of 254 windows — one stretch of the recording around each go cue. A list of arrays is awkward: you cannot do arithmetic on it, and you cannot ask it for a column.

Turn it into a **2-D array** and both problems vanish. What you get is, essentially, a spreadsheet: **one row per trial, one column per moment in time.**

## Our data — 254 trials, 40 frames each

Run this first. It rebuilds what you made by hand at the end of notebook 2.

- `go_windows` — a **list** of 254 one-dimensional arrays
- each is 40 frames long: 10 before the cue, 30 after
- `onsets_of()` and `windows_of()` are the helpers, so you can do the same for `nogo` and `reward`

> Note `windows_of` quietly **skips** any cue too close to the start or end of the recording. Without that guard `dff[-8:32]` returns something — just not what you meant.

In [ ]:
# Run this first. Loads the neuron, then cuts one window around every cue.
import urllib.request
from pathlib import Path

import numpy as np

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab05/data/"

NEURON = "neuron_0588_20180828_5.csv"
if not Path(NEURON).exists():
    urllib.request.urlretrieve(DATA_URL + NEURON, NEURON)

dff, cue, go, nogo, reward = np.loadtxt(
    NEURON, delimiter=",", skiprows=1, usecols=(1, 2, 3, 4, 5), unpack=True
)

FRAME_RATE = 10.8
BEFORE, AFTER = 10, 30      # frames to keep either side of a cue


def onsets_of(marker):
    """The frames where a marker switches 0 -> 1."""
    return np.where(np.diff(marker) == 1)[0] + 1


def windows_of(marker):
    """One window per cue, skipping any that run off either end of the recording."""
    starts = onsets_of(marker)
    return [dff[s - BEFORE:s + AFTER] for s in starts
            if s >= BEFORE and s + AFTER <= len(dff)]


go_windows = windows_of(go)

print("go cues:", len(onsets_of(go)))
print("usable windows:", len(go_windows))
print("each one is", len(go_windows[0]), "frames long")

## Skill 1 — Stack a list of arrays into a table

`np.stack` takes a list of equal-length arrays and lays them out as rows:

```python
trials = np.stack(go_windows)
trials.shape     # (254, 40)
```

Read that shape as **(rows, columns)** = **(trials, frames)**. `.ndim` is now 2.

This only works because every window is the same length. If one were 32 frames long, numpy would refuse — which is a *good* error, far better than a table with a ragged hole in it.

In [ ]:
# DEMO - from list of arrays to one 2-D array
trials = np.stack(go_windows)

print("type: ", type(trials).__name__)
print("shape:", trials.shape, " -> 254 trials, 40 frames each")
print("ndim: ", trials.ndim)
print("total numbers:", trials.size)

### 🧱 DIY challenge

1. Build `nogo_trials` from `windows_of(nogo)` and print its shape.
2. There were 264 nogo cues. How many rows do you get? Why the difference?
3. Build `reward_trials` the same way.

> 💡 Look back at the note under "Our data". The very first nogo cue arrives at frame 2 — only 2 frames into the recording.

In [ ]:
# TODO - your turn

## Skill 2 — Indexing with two numbers

One dimension needed one index. Two dimensions need two, separated by a comma — **row first, then column**:

```python
trials[0]        # the whole first trial   -> 40 numbers
trials[0, 5]     # trial 0, frame 5        -> one number
trials[:, 5]     # frame 5 of EVERY trial  -> 254 numbers
trials[:, :10]   # the baseline of every trial -> (254, 10)
```

That third one is the new idea. `:` means "all of them", so `trials[:, 5]` reads **down a column** — the same instant in time, across every trial.

In [ ]:
# DEMO - rows are trials, columns are moments
print("one trial:        ", trials[0].shape, trials[0][:5].round(3))
print("one value:        ", round(trials[0, 5], 3))
print("frame 5, all trials:", trials[:, 5].shape)
print("first 3 trials, first 4 frames:")
print(trials[:3, :4].round(3))

# the 10 baseline frames of every trial
baseline = trials[:, :BEFORE]
print("baseline block:   ", baseline.shape)

### 🧱 DIY challenge

1. Print the **last** trial's first 5 values.
2. Pull out column 20 (about one second after the cue) for all trials, and print its mean.
3. Print the block of the **first 5 trials** over the **10 frames after the cue** — that is rows `:5` and columns `BEFORE:BEFORE + 10`.

> 💡 `trials[row, column]`. A `:` in either position means "everything in that dimension".

In [ ]:
# TODO - your turn

## Skill 3 — `axis` — which way do you want to collapse?

`trials.mean()` averages all 10,160 numbers into one. Almost never what you want. The `axis` argument says **which dimension to flatten**:

| call | collapses | result | means |
|---|---|---|---|
| `trials.mean(axis=0)` | the rows | 40 numbers | the **average trial** |
| `trials.mean(axis=1)` | the columns | 254 numbers | **each trial's** average |

The rule that actually helps: **`axis=0` runs down the columns, `axis=1` runs along the rows.** The axis you name is the one that disappears.

`axis=0` is the one that matters here. Any single trial is mostly noise; averaging 254 of them cancels the noise and leaves whatever is common to all — the response.

In [ ]:
# DEMO - average down the columns
average_trial = trials.mean(axis=0)
per_trial = trials.mean(axis=1)

print("average trial: ", average_trial.shape, "<- one value per moment in time")
print("per trial:     ", per_trial.shape, "<- one value per trial")
print()
print("the average trial, rounded:")
print(average_trial.round(2))

### 🧱 DIY challenge

1. Print `average_trial[:BEFORE].mean()` — the average level **before** the cue. This is the baseline.
2. Print `average_trial[BEFORE:].mean()` — the average **after**. Is it higher?
3. Use `.argmax()` on `average_trial` to find the peak, then convert it to **seconds relative to the cue**: `(peak - BEFORE) / FRAME_RATE`.
4. Try `trials.max(axis=0)` and `trials.std(axis=0)`. Every summary method takes `axis`.

> 💡 A negative answer in step 3 would mean the peak came *before* the cue — which would be a bad sign for your analysis.

In [ ]:
# TODO - your turn

## Skill 4 — So what is this neuron actually for? 🏆

You now have everything you need to answer the real question. For each of the three events — `go`, `nogo`, `reward` — build the trial table, average down the columns, and compare the level **before** the cue with the **peak** after it.

One neuron, three questions, and the answer is a genuine result about a real mouse.

In [ ]:
# DEMO - the whole analysis, for the go cue
trials = np.stack(windows_of(go))
average_trial = trials.mean(axis=0)

before = average_trial[:BEFORE].mean()
peak = average_trial[BEFORE:].max()
peak_at = (average_trial[BEFORE:].argmax()) / FRAME_RATE

print(f"go   n={trials.shape[0]:3d}  baseline={before:+.3f}  peak={peak:+.3f}  at {peak_at:.1f}s")

### 🏆 DIY finale — three events, one table

1. Wrap the demo in a function `summarise(name, marker)` that prints that same line for any marker.
2. Call it for `go`, `nogo` and `reward`.
3. Read the three lines. **Which events does this neuron respond to, and which does it ignore?**

Then think about what that means. The mouse sees a go cue and a nogo cue — both are a whisker being touched, both are the same physical stimulus. If the neuron responds to one and not the other, it is not reporting *"a whisker was touched"*. It is reporting something about what the touch **means**.

> 💡 You are looking for a baseline near zero and a clearly positive peak. One of the three should be obviously flat.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- Broadcasting — how numpy combines arrays of different shapes: <https://numpy.org/doc/stable/user/basics.broadcasting.html>
- `np.stack`, `np.vstack`, `np.concatenate` — when to use which: <https://numpy.org/doc/stable/reference/routines.array-manipulation.html>
- [ link — Helfrid to add a 2-D / axis explainer ]

## Recap

You can now:

- turn a list of equal-length arrays into a 2-D array with `np.stack`
- read a shape as **(rows, columns)** — here, **(trials, frames)**
- index with `[row, column]`, and use `:` to take a whole row or a whole column
- collapse a dimension with `axis=0` (down the columns) or `axis=1` (along the rows)
- average across trials to pull a signal out of noise — and answer a real question with it

**Next:** `04_vectorisation` — why all of this is also *fast*, and how to stop writing loops.